In [49]:
%%writefile config.py
import os

# ================================
# PROYECTO
# ================================
PROJECT_NAME = "RIS_RAG_Assistant"
VERSION = "1.0.0"

# ================================
# RUTAS DE DIRECTORIOS Y LOGS
# ================================
DATA_DIR = "./data"

CHROMA_DB_DIR = "./chroma_db_ris"
CHROMA_COLLECTION_NAME = "ris_knowledge_base"

LOG_DIR = "./logs"
RUTA_LOGS = "./logs/historial_evaluacion_rag.csv"
FEEDBACK_PATH = "./logs/feedback_usuarios.csv"

# Creación automática de directorios necesarios para evitar runtime errors
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ================================
# DOCUMENTOS FUENTE
# ================================
FICHAS_TECNICAS_PATH = os.path.join(
    DATA_DIR,
    "Fichas_Tecnicas_Materiales_RIS.pdf"
)

MANUALES_PATH = os.path.join(
    DATA_DIR,
    "Manuales_Procesos_Procedimientos_RIS.pdf"
)

MATERIALES_EXCEL_PATH = os.path.join(
    DATA_DIR,
    "Especificaciones_Materiales_RIS.xlsx"
)

EQUIPOS_EXCEL_PATH = os.path.join(
    DATA_DIR,
    "Especificaciones_Equipos_RIS.xlsx"
)

# ================================
# MODELOS IA
# ================================
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
LLM_MODEL_NAME = "gemini-3.5-flash"

# ================================
# CHUNKING
# ================================
CHUNK_SIZE = 650
CHUNK_OVERLAP = 120

IGNORE_TOP_PAGES_MANUAL = 3
IGNORE_TOP_PAGES_FICHAS = 1

# ================================
# RETRIEVAL Y BÚSQUEDA HÍBRIDA
# ================================
THRESHOLD_CONFIANZA = 0.60

TOP_K_BM25 = 20
TOP_K_VECTOR = 20

RRF_K = 60
RRF_TOP_N = 15

TOP_K_FINAL = 3

# ================================
# CONFIGURACIÓN DE EMBEDDINGS
# ================================
EMBEDDING_BATCH_SIZE = 32

# ================================
# REINDEXACIÓN GLOBAL
# ================================
FORCE_REINDEX = False

# ================================
# LLM Y GENERACIÓN
# ================================
LLM_TEMPERATURE = 0.0
MAX_REINTENTOS_API = 2

# Configuración del LLM
LLM_MODEL_NAME = "gemini-3.5-flash"
LLM_TEMPERATURE = 0.0
LLM_MAX_RETRIES = 2

# Umbral de Retrieval
RETRIEVAL_THRESHOLD = 0.50

# ================================
# GEMINI API KEY
# ================================
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

Overwriting config.py


In [36]:
import sys

# Limpiamos el caché del módulo por si teníamos importaciones previas
if 'config' in sys.modules:
    del sys.modules['config']

import config

print(f"✓ [{config.PROJECT_NAME} v{config.VERSION}] Configuración RAG cargada con éxito.")
print(f"• Colección Chroma: {config.CHROMA_COLLECTION_NAME}")
print(f"• Modelo Reranker: {config.RERANKER_MODEL_NAME}")
print(f"• Modelo Gemini: {config.LLM_MODEL_NAME}")
print(f"• Umbral Confianza: {config.THRESHOLD_CONFIANZA}")

✓ [RIS_RAG_Assistant v1.0.0] Configuración RAG cargada con éxito.
• Colección Chroma: ris_knowledge_base
• Modelo Reranker: BAAI/bge-reranker-v2-m3
• Modelo Gemini: gemini-3.5-flash
• Umbral Confianza: 0.6


In [37]:
%%writefile ingestion.py
import os
import re
import pandas as pd
import pypdf
from typing import List, Tuple, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
import config

# ================================
# VALIDACIÓN DE ARCHIVOS
# ================================
def validate_file_path(file_path: str) -> None:
    """Verifica si el archivo existe antes de intentar cargarlo."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"❌ Archivo no encontrado: {file_path}. "
            f"Verifica que el archivo esté presente en la carpeta '{config.DATA_DIR}'."
        )

# ================================
# TOKENIZADOR TÉCNICO UNIFICADO
# ================================
def tokenize_technical_text(text: str) -> List[str]:
    """Preserva acentos, números, caracteres especiales y guiones en términos técnicos."""
    return re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9\-_/\.]+", text.lower())

# ================================
# DICCIONARIO DE CONCEPTOS EQUIVALENTES
# ================================
DICCIONARIO_EXPANSION = {
    "granallado": ["chorro abrasivo", "abrasivo", "sa2.5", "sspc-sp10"],
    "epoxi": ["epoxico", "epo"],
    "pelicula": ["espesor", "mils", "micras", "micronage"],
    "limpieza": ["preparacion", "acondicionamiento"]
}

def expandir_query(query: str) -> str:
    """Enriquece la consulta expandiendo tokens con sus equivalentes técnicos."""
    tokens = tokenize_technical_text(query)
    query_expandida = []

    for token in tokens:
        query_expandida.append(token)
        if token in DICCIONARIO_EXPANSION:
            query_expandida.extend(DICCIONARIO_EXPANSION[token])

    return " ".join(query_expandida)

# ================================
# CARGA DE PDFs CON ENRIQUECIMIENTO CONTEXTUAL
# ================================
def load_and_chunk_pdf(
    pdf_path: str,
    doc_type: str,
    ignore_top_pages: int = 0,
    chunk_size: int = config.CHUNK_SIZE,
    chunk_overlap: int = config.CHUNK_OVERLAP
) -> Tuple[List[str], List[Dict[str, Any]]]:
    """Carga PDF usando pypdf, divide en chunks y genera metadatos enriquecidos."""
    validate_file_path(pdf_path)

    try:
        reader = pypdf.PdfReader(pdf_path)
    except Exception as e:
        print(f"Error al abrir PDF {pdf_path}: {e}")
        return [], []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = []
    metadatos = []
    chunk_counter = 0
    prefix = doc_type.lower().replace(" ", "_")

    for idx in range(ignore_top_pages, len(reader.pages)):
        page_text = reader.pages[idx].extract_text()
        if not page_text or len(page_text.strip()) < 50:
            continue

        page_chunks = text_splitter.split_text(page_text)

        for c in page_chunks:
            chunk_counter += 1
            unique_chunk_id = f"{prefix}_{chunk_counter}"
            text_con_contexto = f"Documento: {doc_type} | Página: {idx + 1}\nContenido:\n{c}"

            chunks.append(text_con_contexto)
            metadatos.append({
                "chunk_id": unique_chunk_id,
                "fuente": pdf_path,
                "tipo_documento": doc_type,
                "pagina": idx + 1,
                "texto_raw": c
            })

    return chunks, metadatos

# ================================
# CARGA DE EXCEL CON METADATOS
# ================================
def load_excel_as_chunks(
    excel_path: str,
    entity_type: str
) -> Tuple[List[str], List[Dict[str, Any]]]:
    """Carga archivos Excel y convierte cada fila en un chunk enriquecido."""
    validate_file_path(excel_path)

    try:
        xls = pd.ExcelFile(excel_path)
        df = pd.read_excel(excel_path, sheet_name=xls.sheet_names[0])
    except Exception as e:
        print(f"Error al abrir Excel {excel_path}: {e}")
        return [], []

    chunks = []
    metadatos = []
    chunk_counter = 0

    for idx, row in df.iterrows():
        chunk_counter += 1
        row_str_list = [f"{col}: {row[col]}" for col in df.columns if pd.notna(row[col])]

        codigo_val = row.get('Código ' + entity_type, idx + 1)
        row_text = f"Ficha de {entity_type} [{codigo_val}]:\n" + " | ".join(row_str_list)

        chunks.append(row_text)
        metadatos.append({
            "chunk_id": f"excel_{entity_type.lower()}_{chunk_counter}",
            "fuente": excel_path,
            "tipo_documento": f"Excel {entity_type}",
            "pagina": "N/A",
            "codigo": str(codigo_val),
            "texto_raw": row_text
        })

    return chunks, metadatos

# ================================
# FUNCIÓN ORQUESTADORA DE INGESTIÓN COMPLETA
# ================================
def load_all_documents() -> Tuple[List[str], List[Dict[str, Any]], List[str]]:
    """Ejecuta la carga masiva utilizando las rutas definidas en config.py."""
    chunks_fichas, meta_fichas = load_and_chunk_pdf(
        config.FICHAS_TECNICAS_PATH, "Ficha Técnica", ignore_top_pages=config.IGNORE_TOP_PAGES_FICHAS
    )
    chunks_manuales, meta_manuales = load_and_chunk_pdf(
        config.MANUALES_PATH, "Manual Operativo", ignore_top_pages=config.IGNORE_TOP_PAGES_MANUAL
    )
    chunks_mat_excel, meta_mat_excel = load_excel_as_chunks(
        config.MATERIALES_EXCEL_PATH, "Material"
    )
    chunks_eq_excel, meta_eq_excel = load_excel_as_chunks(
        config.EQUIPOS_EXCEL_PATH, "Equipo"
    )

    todos_los_chunks = chunks_fichas + chunks_manuales + chunks_mat_excel + chunks_eq_excel
    todos_los_metadatos = meta_fichas + meta_manuales + meta_mat_excel + meta_eq_excel
    todos_los_ids = [m["chunk_id"] for m in todos_los_metadatos]

    print(f"✓ Documentos cargados con éxito: {len(todos_los_chunks)} chunks totales generados.")
    return todos_los_chunks, todos_los_metadatos, todos_los_ids

if __name__ == "__main__":
    chunks, metadata, ids = load_all_documents()
    print("\n===== RESUMEN DE INGESTIÓN =====")
    print(f"• Total Chunks: {len(chunks)}")
    print(f"• Total Metadatos: {len(metadata)}")
    print(f"• Total IDs: {len(ids)}")

Overwriting ingestion.py


In [38]:
!pip install -q pypdf langchain-text-splitters pandas

In [39]:
import sys
import os

if 'ingestion' in sys.modules:
    del sys.modules['ingestion']
if 'config' in sys.modules:
    del sys.modules['config']

import config
import ingestion

print("✓ Configuración de rutas activa.")
print(f"• ¿Existe carpeta data?: {os.path.exists(config.DATA_DIR)}")
print(f"• ¿Existe carpeta logs?: {os.path.exists(config.LOG_DIR)}")
print(f"• Función principal de ingesta: ingestion.load_all_documents")

✓ Configuración de rutas activa.
• ¿Existe carpeta data?: True
• ¿Existe carpeta logs?: True
• Función principal de ingesta: ingestion.load_all_documents


In [40]:
!pip install -q chromadb rank-bm25 sentence-transformers

In [41]:
%%writefile indexer.py
import os
from typing import Dict, Any, List
from rank_bm25 import BM25Okapi
import chromadb
from sentence_transformers import SentenceTransformer

import config
from ingestion import load_all_documents, tokenize_technical_text

# ================================
# MODELO DE EMBEDDINGS (SINGLETON)
# ================================
_embedding_model = None

def get_embedding_model() -> SentenceTransformer:
    """Carga y reutiliza la instancia del modelo SentenceTransformer."""
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(config.EMBEDDING_MODEL_NAME)
    return _embedding_model

# ================================
# CONSTRUCCIÓN DEL ÍNDICE BM25
# ================================
def build_bm25_index(chunks: List[str]) -> BM25Okapi:
    """Tokeniza el corpus usando el tokenizador técnico y genera el índice BM25."""
    tokenized_corpus = [tokenize_technical_text(doc) for doc in chunks]
    return BM25Okapi(tokenized_corpus)

# ================================
# GESTIÓN DE CHROMADB Y RECURSOS
# ================================
def initialize_indexes(force_reindex: bool = config.FORCE_REINDEX) -> Dict[str, Any]:
    """
    Orquesta la inicialización optimizada del RAG:
    - Si la colección existe y no se fuerza reindexación, recupera documentos directamente de ChromaDB.
    - Si se fuerza reindexación o no existe, ejecuta el pipeline de ingestión completo.
    - Retorna un diccionario estructurado de recursos.
    """
    print("Inicializando recursos de indexación...")
    chroma_client = chromadb.PersistentClient(path=config.CHROMA_DB_DIR)
    model = get_embedding_model()

    if force_reindex:
        print(f"Reindexación forzada activa: Eliminando colección '{config.CHROMA_COLLECTION_NAME}'...")
        try:
            chroma_client.delete_collection(config.CHROMA_COLLECTION_NAME)
        except Exception:
            pass

    try:
        collection = chroma_client.get_collection(name=config.CHROMA_COLLECTION_NAME)
        print(f"✓ Colección existente detectada en ChromaDB ({collection.count()} registros).")

        # Recuperamos los documentos guardados para alimentar a BM25 sin releer los archivos
        data = collection.get(include=["documents", "metadatas"])
        chunks = data["documents"]
        metadatos = data["metadatas"]
        ids = data["ids"]

    except Exception:
        print("Colección no encontrada o reindexación requerida. Ejecutando pipeline de ingestión...")
        chunks, metadatos, ids = load_all_documents()

        collection = chroma_client.create_collection(
            name=config.CHROMA_COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )

        passages = [f"passage: {doc}" for doc in chunks]
        embeddings = model.encode(
            passages,
            batch_size=config.EMBEDDING_BATCH_SIZE,
            show_progress_bar=True
        ).tolist()

        collection.add(
            documents=chunks,
            embeddings=embeddings,
            metadatas=metadatos,
            ids=ids
        )
        print("✓ Indexación vectorial en ChromaDB completada con éxito.")

    print("Construyendo/Sincronizando índice lexical (BM25)...")
    bm25 = build_bm25_index(chunks)

    return {
        "bm25": bm25,
        "collection": collection,
        "embedding_model": model,
        "chunks": chunks,
        "metadatos": metadatos,
        "ids": ids
    }

if __name__ == "__main__":
    resources = initialize_indexes()
    print("\n===== RESUMEN DE RECURSOS =====")
    print(f"• Items en BM25: {len(resources['chunks'])}")
    print(f"• Registros en ChromaDB: {resources['collection'].count()}")
    print("✓ Módulo indexer.py perfeccionado.")

Overwriting indexer.py


In [43]:
import sys

if 'indexer' in sys.modules:
    del sys.modules['indexer']

import indexer

rag_resources = indexer.initialize_indexes()

print("\n✓ Recursos vinculados correctamente:")
print(f"• BM25 con {len(rag_resources['chunks'])} chunks.")
print(f"• Colección activa: '{rag_resources['collection'].name}' con {rag_resources['collection'].count()} items.")
print(f"• Modelo vectorial: {config.EMBEDDING_MODEL_NAME}")

Inicializando recursos de indexación...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Colección existente detectada en ChromaDB (800 registros).
Construyendo/Sincronizando índice lexical (BM25)...

✓ Recursos vinculados correctamente:
• BM25 con 800 chunks.
• Colección activa: 'ris_knowledge_base' con 800 items.
• Modelo vectorial: intfloat/multilingual-e5-base


In [44]:
!pip install -q sentence-transformers

In [45]:
%%writefile retrieval.py
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from sentence_transformers import CrossEncoder

import config
from ingestion import tokenize_technical_text, expandir_query

# ================================
# MODELO CROSS-ENCODER (SINGLETON)
# ================================
_reranker_model = None

def get_reranker_model() -> CrossEncoder:
    """Carga y reutiliza la instancia del modelo Reranker."""
    global _reranker_model
    if _reranker_model is None:
        _reranker_model = CrossEncoder(config.RERANKER_MODEL_NAME)
    return _reranker_model

# ================================
# VALIDACIÓN DE CONFIANZA
# ================================
def es_respuesta_valida(candidatos: List[Dict[str, Any]], score_top1: float) -> bool:
    """Valida si el candidato top-1 supera el umbral de confianza calibrado."""
    if not candidatos:
        return False
    if score_top1 < config.THRESHOLD_CONFIANZA:
        return False
    return True

# ================================
# BÚSQUEDA LÉXICA (BM25)
# ================================
def search_bm25(
    query: str,
    bm25_index: Any,
    chunks: List[str],
    metadatos: List[Dict[str, Any]],
    ids: List[str],
    k: int = config.TOP_K_BM25
) -> List[Dict[str, Any]]:
    """Ejecuta búsqueda lexical con BM25 tras expandir y tokenizar la query."""
    query_expandida = expandir_query(query)
    tokenized_query = tokenize_technical_text(query_expandida)
    scores = bm25_index.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "id": ids[idx],
            "document": chunks[idx],
            "metadata": metadatos[idx],
            "bm25_score": float(scores[idx])
        })
    return results

# ================================
# BÚSQUEDA VECTORIAL (CHROMADB)
# ================================
def search_vectorial(
    query: str,
    collection: Any,
    embedding_model: Any,
    k: int = config.TOP_K_VECTOR
) -> List[Dict[str, Any]]:
    """Ejecuta búsqueda semántica en ChromaDB anteponiendo 'query: ' al texto."""
    query_text = f"query: {query}"
    query_embedding = embedding_model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    vector_results = []
    if results and results.get("ids") and len(results["ids"]) > 0:
        for i in range(len(results["ids"][0])):
            vector_results.append({
                "id": results["ids"][0][i],
                "document": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "vector_distance": results["distances"][0][i]
            })
    return vector_results

# ================================
# RECIPROCAL RANK FUSION (RRF)
# ================================
def reciprocal_rank_fusion(
    bm25_results: List[Dict[str, Any]],
    vector_results: List[Dict[str, Any]],
    k_rrf: int = config.RRF_K,
    top_n: int = config.RRF_TOP_N
) -> List[Dict[str, Any]]:
    """Combina y reordena los resultados de BM25 y vectorial usando RRF."""
    rrf_scores = {}
    item_map = {}

    for rank, item in enumerate(bm25_results):
        doc_id = item["id"]
        item_map[doc_id] = item
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

    for rank, item in enumerate(vector_results):
        doc_id = item["id"]
        if doc_id not in item_map:
            item_map[doc_id] = item
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    fused_list = []
    for doc_id, rrf_score in sorted_docs:
        base_item = item_map[doc_id]
        base_item["rrf_score"] = rrf_score
        fused_list.append(base_item)

    return fused_list

# ================================
# ORQUESTADOR DE RETRIEVAL
# ================================
def ejecutar_retrieval_rag(
    query: str,
    rag_resources: Dict[str, Any]
) -> Tuple[List[Dict[str, Any]], float, bool]:
    """
    Ejecuta el pipeline completo de Búsqueda Híbrida + RRF + CrossEncoder Reranker.
    Acepta el diccionario de recursos entregado por indexer.py.
    """
    bm25_index = rag_resources["bm25"]
    collection = rag_resources["collection"]
    embedding_model = rag_resources["embedding_model"]
    chunks = rag_resources["chunks"]
    metadatos = rag_resources["metadatos"]
    ids = rag_resources["ids"]

    # 1. Búsquedas individuales
    res_bm25 = search_bm25(query, bm25_index, chunks, metadatos, ids, k=config.TOP_K_BM25)
    res_vec = search_vectorial(query, collection, embedding_model, k=config.TOP_K_VECTOR)

    # 2. Fusión RRF
    candidatos = reciprocal_rank_fusion(res_bm25, res_vec, k_rrf=config.RRF_K, top_n=config.RRF_TOP_N)

    if not candidatos:
        return [], 0.0, False

    # 3. Reranking con CrossEncoder evaluando contra 'texto_raw'
    reranker_model = get_reranker_model()
    pairs = [[query, doc["metadata"]["texto_raw"]] for doc in candidatos]
    raw_scores = reranker_model.predict(pairs)

    # Normalización sigmoide para convertir a puntaje de confianza [0, 1]
    scores_norm = 1 / (1 + np.exp(-np.array(raw_scores)))

    for i, doc in enumerate(candidatos):
        doc["cross_encoder_raw"] = float(raw_scores[i])
        doc["confidence_score"] = float(scores_norm[i])

    # 4. Ordenamiento final y selección de Top K
    candidatos_ordenados = sorted(candidatos, key=lambda x: x["confidence_score"], reverse=True)[:config.TOP_K_FINAL]
    top_score = candidatos_ordenados[0]["confidence_score"] if candidatos_ordenados else 0.0

    confiable = es_respuesta_valida(candidatos_ordenados, top_score)
    return candidatos_ordenados, top_score, confiable

# ================================
# CONSTRUCCIÓN DE PROMPT LLM
# ================================
def construir_prompt_para_llm(
    pregunta: str,
    documentos_recuperados: List[Dict[str, Any]],
    es_confiable: bool
) -> Tuple[Optional[str], Optional[str]]:
    """Construye el prompt formal para la API del LLM o retorna mensaje de rechazo."""
    if not es_confiable:
        return None, "No se encontró información técnica suficiente en los manuales ni fichas técnicas para responder a esta consulta con la precisión requerida."

    bloques_contexto = []
    for d in documentos_recuperados:
        meta = d["metadata"]
        ref = f"{meta['tipo_documento']} (Página/Código: {meta.get('pagina', meta.get('codigo'))}, ID: {meta['chunk_id']})"
        bloques_contexto.append(f"--- FUENTE: {ref} ---\n{meta['texto_raw']}")

    contexto_str = "\n\n".join(bloques_contexto)

    prompt = f"""
Eres un asistente técnico especializado en recubrimientos industriales RIS.
Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando los fragmentos de contexto técnico proporcionados a continuación.

REGLAS DE GENERACIÓN:
1. Responde de forma técnica, precisa y directa.
2. Para cada afirmación importante, indica explícitamente la fuente utilizada (Ejemplo: [Manual Operativo, Página 6]).
3. Si la información solicitada no está explícitamente en el contexto, indica estrictamente: "No dispongo de información suficiente en la documentación para responder a este punto."

CONTEXTO TÉCNICO:
{contexto_str}

PREGUNTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA:
"""
    return prompt, None

if __name__ == "__main__":
    from indexer import initialize_indexes
    print("Probando retrieval.py de forma autónoma...")
    resources = initialize_indexes()
    q = "¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?"
    docs, score, confiable = ejecutar_retrieval_rag(q, resources)
    print(f"✓ Consulta de prueba ejecutada con éxito. Top-1 score: {score:.4f} | Confiable: {confiable}")

Writing retrieval.py


In [46]:
import sys

# Forzamos recarga del módulo para aplicar cambios
if 'retrieval' in sys.modules:
    del sys.modules['retrieval']

from indexer import initialize_indexes
import retrieval

# 1. Cargamos los recursos indexados
rag_resources = initialize_indexes()

# 2. Pregunta de prueba original
pregunta = "¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?"

# 3. Ejecutamos el pipeline de retrieval
docs, score, es_confiable = retrieval.ejecutar_retrieval_rag(pregunta, rag_resources)
prompt_final, mensaje_error = retrieval.construir_prompt_para_llm(pregunta, docs, es_confiable)

# 4. Imprimimos resultados de prueba
print(f"================ RESULTADO RETRIEVAL ================")
print(f"Pregunta: {pregunta}")
print(f"Confianza Top-1: {score:.4f}")
print(f"¿Pasa validación?: {es_confiable}")

if es_confiable:
    print(f"\n================ PROMPT ENVIADO AL LLM ================")
    print(prompt_final)
else:
    print(f"\n================ RESPUESTA RECHAZADA ================")
    print(mensaje_error)

Inicializando recursos de indexación...
✓ Colección existente detectada en ChromaDB (800 registros).
Construyendo/Sincronizando índice lexical (BM25)...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

================ RESULTADO RETRIEVAL ================
Pregunta: ¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?
Confianza Top-1: 0.7061
¿Pasa validación?: True

================ PROMPT ENVIADO AL LLM ================

Eres un asistente técnico especializado en recubrimientos industriales RIS.
Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando los fragmentos de contexto técnico proporcionados a continuación.

REGLAS DE GENERACIÓN:
1. Responde de forma técnica, precisa y directa.
2. Para cada afirmación importante, indica explícitamente la fuente utilizada (Ejemplo: [Manual Operativo, Página 6]).
3. Si la información solicitada no está explícitamente en el contexto, indica estrictamente: "No dispongo de información suficiente en la documentación para responder a este punto."

CONTEXTO TÉCNICO:
--- FUENTE: Manual Operativo (Página/Código: 6, ID: manual_operativo_12) ---
METAL LIMPIO SIN ÓXIDO:
Procedimiento: Desengrase → Despolvatado → I

In [50]:
%%writefile generator.py
import os
import time
import re
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional
from google import genai
from google.genai import types

import config
from retrieval import ejecutar_retrieval_rag

# =====================================================================
# INICIALIZACIÓN DEL CLIENTE GEMINI
# =====================================================================
def get_gemini_client() -> genai.Client:
    """
    Obtiene el cliente de Gemini priorizando Google Colab userdata.
    Lanza RuntimeError si no se encuentra ninguna clave configurada.
    """
    api_key = None

    # Prioridad 1: Google Colab Userdata
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except Exception:
        pass

    # Prioridad 2: Variable de entorno
    if not api_key:
        api_key = os.environ.get("GEMINI_API_KEY", "").strip()

    # Fallo rápido si no hay API Key
    if not api_key:
        raise RuntimeError(
            "No se encontró la variable GEMINI_API_KEY en 'userdata' ni en variables de entorno. "
            "Asegúrate de configurarla antes de ejecutar el módulo."
        )

    return genai.Client(api_key=api_key)


# =====================================================================
# CONSTRUCTOR DE PROMPTS
# =====================================================================
class PromptBuilder:
    @staticmethod
    def construir_prompt(pregunta: str, documentos_recuperados: List[Dict[str, Any]]) -> Tuple[str, str]:
        """Construye las instrucciones del sistema y el prompt de usuario con deduplicación de contextos."""
        bloques_contexto = []
        vistos = set()

        for d in documentos_recuperados:
            texto = d["metadata"]["texto_raw"].strip()
            if texto in vistos:
                continue
            vistos.add(texto)

            meta = d["metadata"]
            pag_cod = meta.get('pagina', meta.get('codigo', 'N/A'))
            ref = f"{meta['tipo_documento']}, Página/Código {pag_cod}"
            bloques_contexto.append(f"--- FUENTE [{ref}] ---\n{texto}")

        contexto_unificado = "\n\n".join(bloques_contexto)

        prompt_sistema = """Eres un asistente técnico especializado en recubrimientos industriales RIS.

INSTRUCCIONES DE RIGOR TÉCNICO Y FORMATO:
1. Responde únicamente utilizando la información explícita del CONTEXTO TÉCNICO.
2. Para cada afirmación o sección técnica, DEBES citar la fuente utilizando el formato exacto: [Manual Operativo, Página 6] o [Ficha Técnica, Código MAT-01].
3. Si una respuesta requiere combinar varios fragmentos o fuentes, hazlo explícitamente indicando todas las fuentes utilizadas para esa sección.
4. PROHIBIDO UTILIZAR FORMATO LATEX O FÓRMULAS MATEMÁTICAS (No uses $$ o $ para procedimientos ni flechas).
5. Presenta las secuencias, pasos o parámetros mediante listas numeradas o con viñetas en Markdown estándar.
6. Si la información no aparece en el contexto, responde estrictamente: "No dispongo de información suficiente en la documentación técnica para responder a esta pregunta."
7. No utilices conocimiento previo ni asumas datos no documentados."""

        prompt_usuario = f"""CONTEXTO TÉCNICO:
{contexto_unificado}

CONSULTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA:"""

        return prompt_sistema, prompt_usuario


# =====================================================================
# GENERADOR LLM (GEMINI)
# =====================================================================
class LLMGenerator:
    def __init__(self, client_instance: Optional[genai.Client] = None, modelo_fijo: str = config.LLM_MODEL_NAME):
        self.client = client_instance or get_gemini_client()
        self.modelo = modelo_fijo

    def generar(
        self,
        prompt_sistema: str,
        prompt_usuario: str,
        max_intentos: int = getattr(config, 'LLM_MAX_RETRIES', 2)
    ) -> Tuple[Optional[str], str]:
        """Envía la solicitud al modelo Gemini con reintentos para manejo de cuota."""
        for intento in range(max_intentos):
            try:
                response = self.client.models.generate_content(
                    model=self.modelo,
                    contents=prompt_usuario,
                    config=types.GenerateContentConfig(
                        system_instruction=prompt_sistema,
                        temperature=getattr(config, 'LLM_TEMPERATURE', 0.0)
                    )
                )
                if response.text:
                    return response.text.strip(), "ok"

            except Exception as e:
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    tiempo_espera = 10 * (intento + 1)
                    print(f" Ráfaga/Cuota temporal (429). Reintentando en {tiempo_espera}s...")
                    time.sleep(tiempo_espera)
                else:
                    print(f"Error en modelo {self.modelo}: {msg}")
                    break

        return None, "error_api_o_cuota"


# =====================================================================
# VALIDADOR TÉCNICO FLEXIBLE
# =====================================================================
class ResponseValidator:
    @staticmethod
    def validar(respuesta_llm: str, es_confiable_retrieval: bool) -> Tuple[bool, str]:
        """Valida las reglas de negocio y presencia de referencias con un patrón flexible."""
        if not es_confiable_retrieval:
            return False, "retrieval_bajo_umbral"

        if len(respuesta_llm.strip()) < 30:
            return False, "respuesta_demasiado_corta"

        frases_evasivas = [
            "no dispongo de información",
            "no se encuentra en los documentos",
            "información insuficiente"
        ]
        if any(f in respuesta_llm.lower() for f in frases_evasivas):
            return False, "informacion_insuficiente"

        # Regex optimizado y más flexible: acepta citas con/sin corchetes
        patron_citas = r"(?:\[[^\]]*(?:Manual|Ficha|Página|Código|Pág|MAT|EQ)[^\]]*\]|(?:Manual|Ficha|Página|Código|Pág|MAT|EQ)\s*[\w\-]+)"
        tiene_citas = bool(re.search(patron_citas, respuesta_llm, re.IGNORECASE))

        if not tiene_citas:
            return False, "sin_citas_validas"

        return True, "ok"


# =====================================================================
# FORMATEADOR DE RESPUESTAS Y FALLBACKS (DETERMINISTA)
# =====================================================================
class ResponseFormatter:
    @staticmethod
    def obtener_nivel_confianza(score: float) -> str:
        if score >= 0.80:
            return f"Alta ({score:.2f})"
        elif score >= 0.60:
            return f"Media ({score:.2f})"
        else:
            return f"Baja ({score:.2f})"

    @staticmethod
    def generar_fallback(pregunta: str, motivo: str) -> str:
        mensajes = {
            "score_retrieval_bajo": "No se encontraron documentos técnicos con suficiente evidencia en la base de conocimientos para responder esta consulta.",
            "error_api_o_cuota": "La consulta no pudo ser procesada por limitaciones temporales en la API del LLM. Intente de nuevo en un minuto.",
            "sin_citas_validas": "La respuesta generada no incluyó referencias explícitas a la documentación.",
            "informacion_insuficiente": "La información solicitada no está disponible en la documentación técnica actual."
        }
        explicacion = mensajes.get(motivo, "No fue posible procesar la consulta con suficiente evidencia documental.")

        return f"""### Consulta No Completada

> **Consulta:** *"{pregunta}"*

{explicacion}

**Sugerencias:**
• Revisa el manual o ficha técnica correspondiente.
• Consulte directamente con el área técnica responsable.
• Reformule la pregunta con términos más específicos.

---
*Estado de Seguridad:* Rechazado (`Motivo: {motivo}`)"""

    @classmethod
    def dar_formato_final(cls, respuesta_llm: str, documentos_recuperados: List[Dict[str, Any]], score_retrieval: float) -> str:
        # Fuentes 100% deterministas basadas en la metadata de retrieval recuperada
        fuentes_limpias = set()
        for d in documentos_recuperados:
            meta = d["metadata"]
            pag_cod = meta.get('pagina', meta.get('codigo', 'N/A'))
            tipo_doc = meta.get('tipo_documento', 'Documento')
            fuentes_limpias.add(f"• {tipo_doc}, Página/Código {pag_cod}")

        bloque_fuentes = "\n".join(sorted(list(fuentes_limpias)))
        nivel_confianza = cls.obtener_nivel_confianza(score_retrieval)

        return f"""{respuesta_llm}

---
### Fuentes Consultadas
{bloque_fuentes}

**Nivel de Confianza del Retrieval:** `{nivel_confianza}`"""


# =====================================================================
# AGENTE RAG PRINCIPAL CON TRAZABILIDAD
# =====================================================================
class RAGAgent:
    def __init__(self, rag_resources: Dict[str, Any]):
        self.rag_resources = rag_resources
        self.prompt_builder = PromptBuilder()
        self.llm_generator = LLMGenerator()
        self.validator = ResponseValidator()
        self.formatter = ResponseFormatter()
        self.historial_evaluacion: List[Dict[str, Any]] = []

    def responder_consulta(self, pregunta: str) -> str:
        inicio_tiempo = time.time()

        # 1. Retrieval
        candidatos, score, es_confiable = ejecutar_retrieval_rag(pregunta, self.rag_resources)

        ids_chunks = [
            f"{d['metadata'].get('tipo_documento','doc')}_p{d['metadata'].get('pagina', d['metadata'].get('codigo', 'NA'))}"
            for d in candidatos
        ]

        if not es_confiable:
            respuesta_final = self.formatter.generar_fallback(pregunta, "score_retrieval_bajo")
            self._registrar_experimento(pregunta, score, "rechazado_retrieval", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

        # 2. Prompting (Se descarta la variable no usada con '_')
        prompt_sis, prompt_usr = self.prompt_builder.construir_prompt(pregunta, candidatos)

        # 3. LLM Generation
        respuesta_raw, estatus_llm = self.llm_generator.generar(prompt_sis, prompt_usr)

        if estatus_llm != "ok":
            respuesta_final = self.formatter.generar_fallback(pregunta, estatus_llm)
            self._registrar_experimento(pregunta, score, f"fallo_llm_{estatus_llm}", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

        # 4. Validation
        es_valida, motivo = self.validator.validar(respuesta_raw, es_confiable)

        # 5. Output
        if es_valida:
            respuesta_final = self.formatter.dar_formato_final(respuesta_raw, candidatos, score)
            self._registrar_experimento(pregunta, score, "éxito", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final
        else:
            respuesta_final = self.formatter.generar_fallback(pregunta, motivo)
            self._registrar_experimento(pregunta, score, f"rechazado_validación_{motivo}", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

    def _registrar_experimento(self, pregunta: str, score: float, estado: str, ids_chunks: List[str], tiempo_ejecucion: float):
        registro = {
            "Fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
            "Modelo_LLM": self.llm_generator.modelo,
            "Pregunta": pregunta,
            "Score_Retrieval": round(score, 4),
            "Nivel_Confianza": self.formatter.obtener_nivel_confianza(score),
            "Chunks_Consultados": ", ".join(ids_chunks),
            "Estado": estado,
            "Tiempo_Segundos": round(tiempo_ejecucion, 2)
        }
        self.historial_evaluacion.append(registro)

    def exportar_historial(self, ruta_csv: str = "historial_evaluacion_rag.csv"):
        df = pd.DataFrame(self.historial_evaluacion)
        df.to_csv(ruta_csv, index=False)
        print(f"Historial guardado exitosamente en: {ruta_csv}")

Overwriting generator.py


In [51]:
import sys

# Recargar módulos
for mod in ['config', 'ingestion', 'indexer', 'retrieval', 'generator']:
    if mod in sys.modules:
        del sys.modules[mod]

from indexer import initialize_indexes
from generator import RAGAgent

# Inicializar recursos
rag_resources = initialize_indexes()

# Instanciar el agente ajustado
agente_ris = RAGAgent(rag_resources)

# Test 1: Consulta dentro de dominio
print("================ TEST 1: CONSULTA DENTRO DE DOMINIO ================\n")
pregunta_1 = "¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?"
print(agente_ris.responder_consulta(pregunta_1))

print("\n" + "="*70 + "\n")

# Test 2: Consulta fuera de dominio
print("================ TEST 2: CONSULTA FUERA DE DOMINIO ================\n")
pregunta_2 = "¿Cuál es el procedimiento para solicitar viáticos de viaje?"
print(agente_ris.responder_consulta(pregunta_2))

Inicializando recursos de indexación...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ Colección existente detectada en ChromaDB (800 registros).
Construyendo/Sincronizando índice lexical (BM25)...
================ TEST 1: CONSULTA DENTRO DE DOMINIO ================



Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Para el tratamiento de una superficie con **corrosión severa**, el procedimiento, parámetros y preparación técnica requeridos son los siguientes:

### 1. Procedimiento Secuencial
El orden de las etapas de preparación es:
* Desengrase
* Chorro de Arena (SA 2.5)
* Lijado
* Despolvatado
* Inspección

[Manual Operativo, Página/Código 6]

### 2. Parámetros de Control
* **Tiempo estimado:** 24-48 horas [Manual Operativo, Página/Código 6]
* **Rugosidad target (objetivo):** Ra 4.5-6.5 µm [Manual Operativo, Página/Código 6]

### 3. Detalles de Ejecución de las Etapas
De acuerdo con las especificaciones de preparación de la misma sección, se deben seguir estas directrices para las fases de desengrase y lijado:

* **Desengrase Detallado:**
  * Usar desengrasante industrial aprobado (típicamente a base de agua o disolvente).
  * Aplicar con cepillo o trapo limpio.
  * Mantener un tiempo de contacto mínimo de 15 minutos.
  * Remover completamente con agua limpia o disolvente específico.
  * Secar c